In [1]:
# IMPORTS
from graph import app 
import pandas as pd
import os
from tqdm.auto import tqdm
import torch
import time 
from Configuration.models import models #import models to evaluate
from dotenv import load_dotenv
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI
from langchain_neo4j import Neo4jGraph  # for schema
from langchain_ollama import ChatOllama

In [2]:
from dotenv import load_dotenv
import os

from langchain_openai import ChatOpenAI

load_dotenv(override=True)

print("System Initialization")

if not os.getenv("OPENROUTER_API_KEY"):
    print("ERROR: OPENROUTER_API_KEY missing.")
elif not os.getenv("NEO4J_URI"):
    print("ERROR: NEO4J URI missing.")
else:
    print("Core API Keys found.")

# ----------------------------------------
# Model Settings
# ----------------------------------------

#model_name = "meta-llama/llama-3.1-8b-instruct"
#model_name = "meta-llama/llama-3.2-3b-instruct"
model_name = "qwen/qwen-2.5-7b-instruct"

print(f"\nInitializing OpenRouter model: {model_name}...")

try:

    model = ChatOpenAI(
        model=model_name,
        api_key=os.getenv("OPENROUTER_API_KEY"),
        base_url="https://openrouter.ai/api/v1",

        temperature=0.0,
        max_tokens=512,

        extra_body={
            "provider": {
                "allow_fallbacks": True,
            }
        }
    )

    print(f"Model {model_name} loaded successfully.")

except Exception as e:
    print(e)

print("\nSetup Complete! Ready for our agent!")

System Initialization
Core API Keys found.

Initializing OpenRouter model: qwen/qwen-2.5-7b-instruct...
Model qwen/qwen-2.5-7b-instruct loaded successfully.

Setup Complete! Ready for our agent!


In [ ]:
import os
import time
import pandas as pd
import torch
from tqdm.auto import tqdm

def evaluate_model(model_name, model, target_file, save_to_memory=False):
    '''
    Evaluates the model using a specific Excel dataset.
    Uses the database connection information provided per row and passes 'save_to_memory'
    to prevent test data leakage during benchmark runs.
    '''

    # Load target Excel file
    # file_path = f"C:/Users/olgag/Desktop/agentic-memory/CyVerACT/extended_workflow/{target_file}"
    file_path = target_file
    df = pd.read_excel(file_path)

    # Setup output directories
    output_dir = f"Outputs_Evaluation/New_MultiDB_Experiment/{model_name}/"
    os.makedirs(output_dir, exist_ok=True)

    target_name = os.path.splitext(os.path.basename(target_file))[0]

    results_path_pkl = os.path.join(output_dir, f"{target_name}_{model_name}.pkl")
    results_path_xlsx = os.path.join(output_dir, f"{target_name}_{model_name}.xlsx")

    # Resume progress if file already exists
    if os.path.exists(results_path_xlsx):
        print(f"Found existing progress at {results_path_xlsx}. Resuming...")
        df_results = pd.read_excel(results_path_xlsx).astype(object)
    else:
        df_results = df.copy().astype(object)

        new_cols = [
            'generated_cypher', 'answer', 'path', 'output',
            'cypher_errors_history', 'cypher_errors', 'is_KG_Valid_correct',
            'is_KG_Valid_correct_with_full_schema', 'total_tokens',
            'completion_tokens', 'prompt_tokens', 'eval_time_s'
        ]

        for col in new_cols:
            if col not in df_results.columns:
                df_results[col] = None

    config = {
        'configurable': {
            'model': model,
            'model_name': model_name,
            'attempts_w_filtered': 5,
            'total_attempts': 10
        },
        'recursion_limit': 50
    }

    print(f"Starting execution on {target_file} | Memory Saving: {save_to_memory}")

    # Main evaluation loop
    pbar = tqdm(
        df_results.iterrows(),
        total=df_results.shape[0],
        desc="Evaluating queries",
        unit="query"
    )

    for i, row in pbar:
        pbar.set_postfix(
            db=row['database_name'],
            query=i
        )

        # Skip if already computed
        if pd.isna(row['generated_cypher']) or row['generated_cypher'] is None:

            # Identify question column
            question_text = row['question'] if 'question' in row else row['Question']

            # FOR MULTI DBs
            inputs = {
                'question': question_text,
                'database_url': row['url'],
                'database_name': row['database_name'],
                'database_user': row['username'],
                'database_password': row['password'],
                'save_to_memory': save_to_memory
            }

            # FOR MOVIES DB
            # inputs = {
            #     'question': question_text,
            #     'database_url': "neo4j://127.0.0.1:7687",
            #     'database_name': "movies",
            #     'database_user': "neo4j",
            #     'database_password': "agentmemory",
            #     'save_to_memory': save_to_memory  # Memory flag passed to graph
            # }

            result = None
            max_retries = 3

            # Robust Execution Retry Loop
            for attempt in range(max_retries):
                try:
                    start = time.perf_counter()
                    result = app.invoke(inputs, config=config)
                    elapsed = time.perf_counter() - start
                    break
                except Exception as e:
                    print(f"\nError on query {i} (Attempt {attempt+1}): {e}")
                    time.sleep(20)

            if result is None:
                continue

            # Extract Graph State Results
            path_list = result.get('path', [])
            is_valid_correct = result.get('is_KG_Valid_correct', False)

            is_full_schema_correct = ('schema_retriever' in path_list) and bool(is_valid_correct)

            # Store All Metrics
            df_results.at[i, 'eval_time_s'] = round(elapsed, 4)
            df_results.at[i, 'generated_cypher'] = result.get('cypher_statement', '')
            df_results.at[i, 'output'] = str(result.get('database_records', ''))
            df_results.at[i, 'path'] = str(path_list)
            df_results.at[i, 'cypher_errors'] = str(result.get('cypher_errors', []))
            df_results.at[i, 'cypher_errors_history'] = str(result.get('cypher_errors_history', []))
            df_results.at[i, 'is_KG_Valid_correct'] = is_valid_correct
            df_results.at[i, 'is_KG_Valid_correct_with_full_schema'] = is_full_schema_correct
            df_results.at[i, 'answer'] = result.get('answer', '')
            df_results.at[i, 'total_tokens'] = str(result.get('total_tokens', ''))
            df_results.at[i, 'prompt_tokens'] = str(result.get('prompt_tokens', ''))
            df_results.at[i, 'completion_tokens'] = str(result.get('completion_tokens', ''))

            # Memory Cleanup
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            # Checkpoint Saves
            df_results.to_pickle(results_path_pkl)
            df_results.to_excel(results_path_xlsx, index=False)

            time.sleep(1) # Sleep delay for stability

        else:
            print(f"Skipping row {i}, already computed.")

    return df_results

In [ ]:
# POPULATE MEMORY (Memory Dataset) ---
model_name_mem = 'llama3_1_8b_'

df_memory_results = evaluate_model(
    model_name=model_name_mem, 
    model=model, 
    target_file='Movies_DBs_Datasets/50_memory_queries_thesis.xlsx',  
    save_to_memory=True                        
)

In [ ]:
# Testing Dataset
model_name_mem = 'qwen2_5_7b_BASELINE'

df_memory_results = evaluate_model(
    model_name=model_name_mem, 
    model=model, 
    target_file='Movies_DBs_Datasets/100_test_queries_thesis.xlsx',  
    save_to_memory=False                        
)

In [ ]:
def evaluate_model (model_name, model, dataset):
    ''' 
    Evaluate the selected model on the given dataset, by invoking the graph workflow.
    Save the dataframe with the questions and the generated cypher queries to the Outputs_Evaluation folder.
    Return the dataframe
    '''
    # torch.cuda.empty_cache()#clear gpu cache
    # Get datasets 
    if dataset =='sustain':
        df = pd.read_excel('../data/text2cypher-sustaingraph/sustaintest.xlsx',sheet_name='Queries')
        subfolder = 'SustainDataset'
    elif dataset == 'benchmark2025':
        df = pd.read_excel('../data/text2cypher-2025v1/test.xlsx')
        #update df only with rows with a database access
        df = df[df['url'].notna()].reset_index(drop=True)
        subfolder = 'BenchmarkDataset2025'
    elif dataset == 'benchmark2024':
        df = pd.read_excel('../data/text2cypher-2024v1/test.xlsx')
        #update df only with rows with a database access
        df = df[df['url'].notna()].reset_index(drop=True)
        subfolder = 'BenchmarkDataset2024'
    
    output_dir = f"Outputs_Evaluation/{subfolder}/{model_name}/"
    # Create the directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)

    results_path_pkl = os.path.join(output_dir, f"{model_name}.pkl")
    results_path_xlsx = os.path.join(output_dir, f"{model_name}.xlsx")
   
    # Check if results file already exists
    if os.path.exists(results_path_xlsx):
        df_results = pd.read_excel(results_path_xlsx)
    else:# Run model on dataset
        df_results = df.copy() # Copy of the df with questions and expected answers, to add the answers of the llm 
        # Now define additional columns in df_results to store LLM results:
        df_results['generated_cypher'] = None 
        df_results['answer'] = None
        df_results['path'] = None           # Use None so we can store a list
        df_results['output'] = None           # Use None so we can store a list
        df_results['cypher_errors_history'] = None           # Use None so we can store a list
        df_results['cypher_errors'] = None           # Use None so we can store a list
        df_results['is_KG_Valid_correct'] = None           # Use None so we can store a list
        df_results['total_tokens'] = None           # Use None so we can store a list
        df_results['completion_tokens'] = None           # Use None so we can store a list
        df_results['prompt_tokens'] = None           # Use None so we can store a list

    config = {'configurable': {'model': model, 'model_name':model_name, 'attempts_w_filtered':5, 'total_attempts':10},'recursion_limit':50}
    
    print(f'Starting evaluation for {model_name} on {dataset} ...')
    # Iterate over each row and compute only if not already done
    for i, row in tqdm(df_results.iterrows(), total=df_results.shape[0]):
        if pd.isna(row['generated_cypher']):
            question = row['Question'] if dataset == 'sustain' else row['question']
            print(i,question)
            
            # Get credentials
            required_cols = {'database_name', 'url', 'username', 'password'}
            if required_cols.issubset(df_results.columns):
                database_url = row['url']
                database_name = row['database_name']
                database_user = row['username']
                database_password = row['password']
            else:
                #its the sustain dataset
                database_url = os.getenv('database_url')
                database_name = os.getenv('database_name')
                database_user = os.getenv('database_user')
                database_password = os.getenv('database_password')
                # activate it for gemma 9b
                # if database_name == 'offshoreleaks':
                #     print(f'Skipping row {i}, to avoid memory error in {database_name}')
                #     continue
                # else:# Configuration 
                inputs = {
                    'question':question,
                    'database_url': database_url,
                    'database_name':database_name,
                    'database_user':database_user,
                    'database_password':database_password
                }
                # torch.cuda.empty_cache() #clear gpu cache
            
                # Run the graph and save outputs
                result =app.invoke(inputs, config =config) #,debug=False
                df_results.at[i, 'generated_cypher'] = result['cypher_statement']
                df_results.at[i, 'output'] = str(result['database_records'])
                df_results.at[i, 'path'] = result['path']
                df_results.at[i, 'cypher_errors'] = result['cypher_errors']
                df_results.at[i, 'cypher_errors_history'] = result['cypher_errors_history']
                df_results.at[i, 'is_KG_Valid_correct'] = result['is_KG_Valid_correct']
                df_results.at[i, 'answer'] = result['answer']
                df_results.at[i, 'total_tokens'] = result['total_tokens']
                df_results.at[i, 'completion_tokens'] = result['completion_tokens']
                df_results.at[i, 'prompt_tokens'] = result['prompt_tokens']
                if ('schema_retriever' in result['path']) and result['is_KG_Valid_correct']:
                    df_results.at[i, 'is_KG_Valid_correct_with_full_schema'] = True
                else:
                    df_results.at[i, 'is_KG_Valid_correct_with_full_schema'] = False
                torch.cuda.empty_cache()#clear gpu cache
                
                # Save results
                df_results.to_pickle(results_path_pkl)
                df_results.to_excel(results_path_xlsx, index=False)# Set index=False to exclude the index column in Excel
        else:
            print(f"Skipping row {i}, already computed.")
            
    return df_results

In [ ]:
model_name = 'llama3_2_3b'
df_results = evaluate_model (model_name, models[model_name](), 'benchmark2025')

# Evaluate with computational metrics

In [2]:
def evaluate_model_usage (model_name, model, dataset):
    ''' 
    Evaluate the selected model on the given dataset, by invoking the graph workflow.
    Save the dataframe with the questions and the generated cypher queries to the Outputs_Evaluation folder.
    Return the dataframe
    '''
    # torch.cuda.empty_cache()#clear gpu cache
    # Get datasets 
    if dataset =='sustain':
        df = pd.read_excel('../data/text2cypher-sustaingraph/sustaintest.xlsx',sheet_name='Queries')
        subfolder = 'SustainDataset'
    elif dataset == 'benchmark2025':
        df = pd.read_excel('../data/text2cypher-2025v1/test.xlsx')
        #update df only with rows with a database access
        df = df[df['url'].notna()].reset_index(drop=True)
        subfolder = 'BenchmarkDataset2025'
    elif dataset == 'benchmark2024':
        df = pd.read_excel('../data/text2cypher-2024v1/test.xlsx')
        #update df only with rows with a database access
        df = df[df['url'].notna()].reset_index(drop=True)
        subfolder = 'BenchmarkDataset2024'
    
    output_dir = f"Outputs_Evaluation/{subfolder}/{model_name}/"
    # Create the directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)

    results_path_pkl = os.path.join(output_dir, f"{model_name}_usage1.pkl")
    results_path_xlsx = os.path.join(output_dir, f"{model_name}_usage1.xlsx")
   
    # Check if results file already exists
    if os.path.exists(results_path_xlsx):
        df_results = pd.read_excel(results_path_xlsx)
    else:# Run model on dataset
        df_results = df.copy() # Copy of the df with questions and expected answers, to add the answers of the llm 
        # Now define additional columns in df_results to store LLM results:
        df_results['generated_cypher'] = None 
        df_results['answer'] = None
        df_results['path'] = None           # Use None so we can store a list
        df_results['output'] = None           # Use None so we can store a list
        df_results['cypher_errors_history'] = None           # Use None so we can store a list
        df_results['cypher_errors'] = None           # Use None so we can store a list
        df_results['is_KG_Valid_correct'] = None           # Use None so we can store a list
        # timing columns
        df_results['eval_time_s'] = None              # <-- total wall-clock time per query
        # tokens columns
        df_results['total_tokens'] = None           # Use None so we can store a list
        df_results['completion_tokens'] = None           # Use None so we can store a list
        df_results['prompt_tokens'] = None           # Use None so we can store a list
    config = {'configurable': {'model': model, 'model_name':model_name, 'attempts_w_filtered':5, 'total_attempts':10},'recursion_limit':50}
    
    print(f'Starting evaluation for {model_name} on {dataset} ...')
    # Iterate over each row and compute only if not already done
    for i, row in tqdm(df_results.iterrows(), total=df_results.shape[0]):
        if pd.isna(row['generated_cypher']):
            question = row['Question'] if dataset == 'sustain' else row['question']
            print(i,question)
            
            # Get credentials
            required_cols = {'database_name', 'url', 'username', 'password'}
            if required_cols.issubset(df_results.columns):
                database_url = row['url']
                database_name = row['database_name']
                database_user = row['username']
                database_password = row['password']
            else:
                #its the sustain dataset
                database_url = os.getenv('database_url')
                database_name = os.getenv('database_name')
                database_user = os.getenv('database_user')
                database_password = os.getenv('database_password')
                # activate it for gemma 9b
                # if database_name == 'offshoreleaks':
                #     print(f'Skipping row {i}, to avoid memory error in {database_name}')
                #     continue
                # else:# Configuration 
                inputs = {
                    'question':question,
                    'database_url': database_url,
                    'database_name':database_name,
                    'database_user':database_user,
                    'database_password':database_password
                }
                # torch.cuda.empty_cache() #clear gpu cache
            
                # Run the graph and save outputs
                start = time.perf_counter() # start wall-clock timer
                result =app.invoke(inputs, config =config) #,debug=False
                elapsed = time.perf_counter() - start # elapsed wall-clock time
                df_results.at[i, 'eval_time_s'] = round(elapsed, 4)
                df_results.at[i, 'generated_cypher'] = result['cypher_statement']
                df_results.at[i, 'output'] = str(result['database_records'])
                df_results.at[i, 'path'] = result['path']
                df_results.at[i, 'cypher_errors'] = result['cypher_errors']
                df_results.at[i, 'cypher_errors_history'] = result['cypher_errors_history']
                df_results.at[i, 'is_KG_Valid_correct'] = result['is_KG_Valid_correct']
                df_results.at[i, 'answer'] = result['answer']
                df_results.at[i, 'total_tokens'] = result['total_tokens']
                df_results.at[i, 'completion_tokens'] = result['completion_tokens']
                df_results.at[i, 'prompt_tokens'] = result['prompt_tokens']
                if ('schema_retriever' in result['path']) and result['is_KG_Valid_correct']:
                    df_results.at[i, 'is_KG_Valid_correct_with_full_schema'] = True
                else:
                    df_results.at[i, 'is_KG_Valid_correct_with_full_schema'] = False
                torch.cuda.empty_cache()#clear gpu cache
                

                # Save results
                df_results.to_pickle(results_path_pkl)
                df_results.to_excel(results_path_xlsx, index=False)# Set index=False to exclude the index column in Excel
        else:
            print(f"Skipping row {i}, already computed.")
            
    return df_results